# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [1]:
team_setup = {
    'team_name': 'Andrew and Anh',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Andrew', 'github': 'hyunderscore'},
        {'name': 'Anh', 'github': 'anhnbach'},
    ],
    'repo_url': 'https://github.com/hyunderscore/walmart-hurricane-project',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/hyunderscore/walmart-hurricane-project/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Andrew',           # nice to have
    'meeting_time': '5 PM on Wednesdays',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['timestamp has at least 4 formats and 886 missing values',
        'store_id has 60 variants for about 14 stores (FL-412, FL_412, fl-412, FL412)',
        'category has 37 values for about 9 categories (EMERGENCY, emergency, Emergency Supplies)',
        'quantity has 130 negative values (-5 to -1)',
        'unit_price is stored as text and 3,029 values start with $'],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  done - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [2]:
import subprocess

REPO_PATH = '/Users/anhbach/Desktop/ds2002-midterm/walmart-hurricane-project'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

33f4449 finished charter.md
ded7151 added initial files


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [3]:
# TODO
def extract():
    return

### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [4]:
# TODO: print or display your decision log

### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [5]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=28.54, lon=-81.38, tries=3):
    """Daily precipitation, max wind speed, and mean sea-level pressure for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'precipitation_sum,windspeed_10m_max,pressure_msl_mean',
        'timezone': 'America/New_York',
    }
    for attempt in range(tries):
        r = requests.get(BASE, params=params, timeout=10)

        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])

        if r.status_code not in RETRY_STATUSES:
            r.raise_for_status()

        time.sleep(attempt + 1)

    raise RuntimeError('all tries failed')


wx = get_weather('2024-09-01', '2024-09-30')

wx = wx.rename(columns={
    'time': 'date',
    'precipitation_sum': 'precip_mm',
    'windspeed_10m_max': 'wind_max_kmh',
    'pressure_msl_mean': 'pressure_hpa',
})
wx['date'] = pd.to_datetime(wx['date'])
wx['source'] = 'open-meteo'

print('rows:', len(wx))
wx.head()

rows: 30


,date,precip_mm,wind_max_kmh,pressure_hpa,source
0,2024-09-01,3.0,18.8,1014.1,open-meteo
1,2024-09-02,12.4,13.5,1014.1,open-meteo
2,2024-09-03,24.8,22.8,1015.6,open-meteo
3,2024-09-04,22.1,15.5,1017.7,open-meteo
4,2024-09-05,22.7,15.2,1017.3,open-meteo


**Source:** Open-Meteo, `https://archive-api.open-meteo.com/v1/archive`. Orlando, Sept 1 to 30, 2024.

**Retries:** tries up to 3 times, retrying on 429 and 500, 502, 503, 504 errors.

**Timezone:** we set `timezone='America/New_York'`, so the weather days match the sales days.

**Lining up with sales:** we will turn sales timestamps into dates with `.dt.normalize()` and left join the weather on `date`.

**Status:** the API call works. 

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [6]:
# TODO

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [7]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': None,        # an integer
    'decisions_logged': None,           # how many entries in your log
    'canonical_products': None,         # distinct products after consolidation
    'weather_api_rows': len(wx),           # rows returned by your API call
    'notebook_runs_clean': False,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  not yet - rows_after_cleaning = None
  not yet - decisions_logged = None
  not yet - canonical_products = None
  done - weather_api_rows = 30
  not yet - notebook_runs_clean = False

required items complete: True
progress items done: 1 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** _..._

**Plan, with an owner and a date:** _..._

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | In progress  | Andrew |
| Q2 SKU consolidation | In progress | Andrew |
| Q3 weather correlation | In progress | Anh |
| Q4 store and geography | Not started | Anh |
| Q5 signal or artifact | Not started | Anh |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.